In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

class NaiveBayes:
    def __init__(self):
        """Initializes all the necessary model parameters."""
        self.class_priors = None
        self.word_lhoods = None
        self.vocab = None
        self.smoothening = None
        self.classes_ = None
        self.class_map_ = None
        self.num_classes = None
        self.vocab_index = None
        
    def fit(self, df, smoothening, class_col="Class Index", text_col="Tokenized Description"):
        """
        Learn the parameters of the model from the training data.
        Classes are 1-indexed.

        Args:
            df (pd.DataFrame): The training data containing columns class_col and text_col.
                each entry of text_col is a list of tokens.
            smoothening (float): The Laplace smoothening parameter.
        """
        self.smoothening = smoothening
        
        # Create a mapping from class labels to 0-based indices
        self.classes_ = np.sort(df[class_col].unique())
        self.num_classes = len(self.classes_)
        self.class_map_ = {label: i for i, label in enumerate(self.classes_)}
        
        # Calculate class priors for value_counts
        class_cnts = df[class_col].value_counts().reindex(self.classes_).fillna(0)
        self.class_priors = np.log(class_cnts / len(df))
        
        # Build the vocabulary
        print("Building vocabulary...")
        self.vocab = set(word for tokens_list in df[text_col] for word in tokens_list)
        self.vocab = sorted(list(self.vocab), key=str) # key=str handles bigrams (tuples)
        self.vocab_index = {word: i for i, word in enumerate(self.vocab)}
        vocab_size = len(self.vocab)
        
        # Tally word counts for each class
        word_counts_per_class = np.zeros((self.num_classes, vocab_size))
        total_words_per_class = np.zeros(self.num_classes)
        
        for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Fitting Model"):
            label = row[class_col]
            class_idx = self.class_map_[label]
            tokens = row[text_col]
            
            total_words_per_class[class_idx] += len(tokens)
            for token in tokens:
                if token in self.vocab_index:
                    token_idx = self.vocab_index[token]
                    word_counts_per_class[class_idx, token_idx] += 1
        
        # Calculate smoothed log-likelihoods
        numerator = word_counts_per_class + self.smoothening
        denominator = total_words_per_class + self.smoothening * vocab_size
        self.word_lhoods = np.log(numerator / denominator[:, np.newaxis])

    def predict(self, df, text_col="Tokenized Description", predicted_col="Predicted"):
        """
        Predict the class of the input data by filling up column predicted_col in the input dataframe.

        Args:
            df (pd.DataFrame): The testing data containing column text_col.
                each entry of text_col is a list of tokens.
        """
        predictions = []
        
        for tokens in tqdm(df[text_col], desc="Predicting Classes"):
            # Start with the log priors for each class
            log_posterior = self.class_priors.copy()
            
            # Add the log likelihoods for each token
            for token in tokens:
                if token in self.vocab_index:
                    token_idx = self.vocab_index[token]
                    log_posterior += self.word_lhoods[:, token_idx]
            
            # Find the index of the class with the highest probability
            predicted_index = np.argmax(log_posterior)
            # Map the index back to the original class label
            predictions.append(self.classes_[predicted_index])
        
        df[predicted_col] = predictions
        
    def accuracy(self, df, class_col="Class Index", predicted_col="Predicted"):
        """
        Computes the accuracy of the predictions.
        """
        correct = (df[class_col] == df[predicted_col]).sum()
        total = len(df)
        return correct / total if total > 0 else 0

In [ ]:
!pip install wordcloud

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.util import bigrams
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Download NLTK data
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.word_tokenize("test")
except LookupError:
    nltk.download('punkt')

# Initialize tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# Data loading
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
print(train_data.head())
print(train_data.isnull().sum())



In [ ]:
# Preprocessing function implementation
def preprocess_text(text, remove_stopwords=False, use_stemming=False):
    tokens = nltk.word_tokenize(text.lower())
    if remove_stopwords:
        tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    if use_stemming:
        tokens = [stemmer.stem(word) for word in tokens]
    return tokens

# Function to add bigrams to a list of tokens
def add_bigrams_features(tokens):
    return tokens + list(bigrams(tokens))

In [ ]:
# Define the 8 model configurations to test
configurations = [
    {'name': 'Unigrams (No pre-processing)', 'sw': False, 'stem': False, 'bigrams': False},
    {'name': 'Unigrams (Stopword removal only)', 'sw': True, 'stem': False, 'bigrams': False},
    {'name': 'Unigrams (Stemming only)', 'sw': False, 'stem': True, 'bigrams': False},
    {'name': 'Unigrams (Stopword removal + stemming)', 'sw': True, 'stem': True, 'bigrams': False},
    {'name': 'Unigrams + Bigrams (No pre-processing)', 'sw': False, 'stem': False, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stopword removal only)', 'sw': True, 'stem': False, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stemming only)', 'sw': False, 'stem': True, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stopword removal + stemming)', 'sw': True, 'stem': True, 'bigrams': True}
]

all_results = []
avg_method = 'weighted'

for config in configurations:
    print(f"\n--- Processing Model: {config['name']} ---")
    
    # Preprocess data based on the configuration
    train_data['tokens'] = train_data['content'].apply(lambda x: preprocess_text(x, remove_stopwords=config['sw'], use_stemming=config['stem']))
    test_data['tokens'] = test_data['content'].apply(lambda x: preprocess_text(x, remove_stopwords=config['sw'], use_stemming=config['stem']))
    
    # Add bigrams if required
    if config['bigrams']:
        train_data['tokens'] = train_data['tokens'].apply(add_bigrams_features)
        test_data['tokens'] = test_data['tokens'].apply(add_bigrams_features)
        
    # Train the Naive Bayes model
    model = NaiveBayes()
    model.fit(train_data, smoothening=1.0, class_col='label', text_col='tokens')
    
    # Predict on train and test sets
    model.predict(train_data, text_col='tokens', predicted_col='Predicted_Train')
    print("Predicting on test data.....")
    model.predict(test_data, text_col='tokens', predicted_col='Predicted_Test')
    
    # Calculate metrics and store them
    y_train_true, y_train_pred = train_data['label'], train_data['Predicted_Train']
    y_test_true, y_test_pred = test_data['label'], test_data['Predicted_Test']
    
    train_metrics = {
        'Accuracy': accuracy_score(y_train_true, y_train_pred),
        'Precision': precision_score(y_train_true, y_train_pred, average=avg_method, zero_division=0),
        'Recall': recall_score(y_train_true, y_train_pred, average=avg_method, zero_division=0),
        'F1-score': f1_score(y_train_true, y_train_pred, average=avg_method, zero_division=0)
    }
    test_metrics = {
        'Accuracy': accuracy_score(y_test_true, y_test_pred),
        'Precision': precision_score(y_test_true, y_test_pred, average=avg_method, zero_division=0),
        'Recall': recall_score(y_test_true, y_test_pred, average=avg_method, zero_division=0),
        'F1-score': f1_score(y_test_true, y_test_pred, average=avg_method, zero_division=0)
    }
    all_results.append({'Model': config['name'], 'Train': train_metrics, 'Test': test_metrics})

In [ ]:
# Create the training data performance table
train_df = pd.DataFrame([{'Model': r['Model'], **r['Train']} for r in all_results]).set_index('Model')

# Create the test data performance table
test_df = pd.DataFrame([{'Model': r['Model'], **r['Test']} for r in all_results]).set_index('Model')

print("Table 1: Performance comparison on training data")
display(train_df.style.format('{:.5f}'))

print("\nTable 2: Performance comparison on test data")
display(test_df.style.format('{:.5f}'))

In [ ]:
import matplotlib.pyplot as plt

remove_stopwords = True
use_stemming = True
bigram = True

print("Generating tokens for word cloud visualization...")
train_data['tokens_for_wc'] = train_data['content'].apply(lambda x: preprocess_text(x, remove_stopwords=remove_stopwords, use_stemming=use_stemming))

# Create a word cloud for each class
unique_labels = sorted(train_data['label'].unique())
num_labels = len(unique_labels)

if bigram:
    train_data['tokens_for_wc'] = train_data['tokens_for_wc'].apply(add_bigrams_features)

cols = 4
rows = (num_labels + cols - 1) // cols
plt.figure(figsize=(20, 5 * rows))

print("Generating word clouds...")
for i, label in enumerate(unique_labels):
    class_tokens_series = train_data[train_data['label'] == label]['tokens_for_wc']
    def format_token(token):
        if isinstance(token, tuple):
            return '_'.join(token) 
        else:
            return token

    full_text = ' '.join([' '.join([format_token(t) for t in token_list]) for token_list in class_tokens_series])
    
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis', collocations=False).generate(full_text)
    
    plt.subplot(rows, cols, i + 1)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Class {label}')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

remove_stopwords = True
use_stemming = True
bigram = True

print("Generating tokens for word cloud visualization...")
test_data['tokens_for_wc'] = test_data['content'].apply(lambda x: preprocess_text(x, remove_stopwords=remove_stopwords, use_stemming=use_stemming))

# Create a word cloud for each class
unique_labels = sorted(test_data['label'].unique())
num_labels = len(unique_labels)

if bigram:
    test_data['tokens_for_wc'] = test_data['tokens_for_wc'].apply(add_bigrams_features)

cols = 4
rows = (num_labels + cols - 1) // cols
plt.figure(figsize=(20, 5 * rows))

print("Generating word clouds...")
for i, label in enumerate(unique_labels):
    class_tokens_series = train_data[train_data['label'] == label]['tokens_for_wc']
    def format_token(token):
        if isinstance(token, tuple):
            return '_'.join(token) 
        else:
            return token

    full_text = ' '.join([' '.join([format_token(t) for t in token_list]) for token_list in class_tokens_series])
    
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis', collocations=False).generate(full_text)
    
    plt.subplot(rows, cols, i + 1)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Class {label}')

plt.tight_layout()
plt.show()

In [ ]:
# Define the 8 model configurations to test
configurations = [
    {'name': 'Unigrams (No pre-processing)', 'sw': False, 'stem': False, 'bigrams': False},
    {'name': 'Unigrams (Stopword removal only)', 'sw': True, 'stem': False, 'bigrams': False},
    {'name': 'Unigrams (Stemming only)', 'sw': False, 'stem': True, 'bigrams': False},
    {'name': 'Unigrams (Stopword removal + stemming)', 'sw': True, 'stem': True, 'bigrams': False},
    {'name': 'Unigrams + Bigrams (No pre-processing)', 'sw': False, 'stem': False, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stopword removal only)', 'sw': True, 'stem': False, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stemming only)', 'sw': False, 'stem': True, 'bigrams': True},
    {'name': 'Unigrams + Bigrams (Stopword removal + stemming)', 'sw': True, 'stem': True, 'bigrams': True}
]

all_results = []
avg_method = 'weighted'

for config in configurations:
    print(f"\n--- Processing Model: {config['name']} ---")
    
    # Preprocess data based on the configuration
    train_data['title_tokens'] = train_data['title'].apply(lambda x: preprocess_text(x, remove_stopwords=config['sw'], use_stemming=config['stem']))
    test_data['title_tokens'] = test_data['title'].apply(lambda x: preprocess_text(x, remove_stopwords=config['sw'], use_stemming=config['stem']))
    
    # Add bigrams if required
    if config['bigrams']:
        train_data['title_tokens'] = train_data['title_tokens'].apply(add_bigrams_features)
        test_data['title_tokens'] = test_data['title_tokens'].apply(add_bigrams_features)
        
    # Train the Naive Bayes model
    model = NaiveBayes()
    model.fit(train_data, smoothening=1.0, class_col='label', text_col='title_tokens')
    
    # Predict on train and test sets
    model.predict(train_data, text_col='title_tokens', predicted_col='Predicted_Train')
    print("Predicting on test data.....")
    model.predict(test_data, text_col='title_tokens', predicted_col='Predicted_Test')
    
    # Calculate metrics and store them
    y_train_true, y_train_pred = train_data['label'], train_data['Predicted_Train']
    y_test_true, y_test_pred = test_data['label'], test_data['Predicted_Test']
    
    train_metrics = {
        'Accuracy': accuracy_score(y_train_true, y_train_pred),
        'Precision': precision_score(y_train_true, y_train_pred, average=avg_method, zero_division=0),
        'Recall': recall_score(y_train_true, y_train_pred, average=avg_method, zero_division=0),
        'F1-score': f1_score(y_train_true, y_train_pred, average=avg_method, zero_division=0)
    }
    test_metrics = {
        'Accuracy': accuracy_score(y_test_true, y_test_pred),
        'Precision': precision_score(y_test_true, y_test_pred, average=avg_method, zero_division=0),
        'Recall': recall_score(y_test_true, y_test_pred, average=avg_method, zero_division=0),
        'F1-score': f1_score(y_test_true, y_test_pred, average=avg_method, zero_division=0)
    }
    all_results.append({'Model': config['name'], 'Train': train_metrics, 'Test': test_metrics})

In [ ]:
# Create the training data performance table
train_df = pd.DataFrame([{'Model': r['Model'], **r['Train']} for r in all_results]).set_index('Model')

# Create the test data performance table
test_df = pd.DataFrame([{'Model': r['Model'], **r['Test']} for r in all_results]).set_index('Model')

# Display the final tables, formatted to 5 decimal places like your example
print("Table 1: Performance comparison on training data")
display(train_df.style.format('{:.5f}'))

print("\nTable 2: Performance comparison on test data")
display(test_df.style.format('{:.5f}'))

In [ ]:
print("Training Content-Only Model")
# Preprocess the content column
train_data['content_tokens'] = train_data['content'].apply(preprocess_text).apply(add_bigrams_features)
test_data['content_tokens'] = test_data['content'].apply(preprocess_text).apply(add_bigrams_features)

# Train and evaluate
model_content = NaiveBayes()
model_content.fit(train_data, smoothening=1.0, class_col='label', text_col='content_tokens')
model_content.predict(train_data, text_col='content_tokens', predicted_col='pred_train')
model_content.predict(test_data, text_col='content_tokens', predicted_col='pred_test')

acc_train_content = model_content.accuracy(train_data, class_col='label', predicted_col='pred_train')
acc_test_content = model_content.accuracy(test_data, class_col='label', predicted_col='pred_test')
print(f"Content-Only Test Accuracy: {acc_test_content:.4f}")

print("\nTraining Title-Only Model")
# Preprocess the title column
train_data['title_tokens'] = train_data['title'].apply(preprocess_text).apply(add_bigrams_features)
test_data['title_tokens'] = test_data['title'].apply(preprocess_text).apply(add_bigrams_features)

# Train and evaluate
model_title = NaiveBayes()
model_title.fit(train_data, smoothening=1.0, class_col='label', text_col='title_tokens')
model_title.predict(train_data, text_col='title_tokens', predicted_col='pred_train')
model_title.predict(test_data, text_col='title_tokens', predicted_col='pred_test')

acc_train_title = model_title.accuracy(train_data, class_col='label', predicted_col='pred_train')
acc_test_title = model_title.accuracy(test_data, class_col='label', predicted_col='pred_test')
print(f"Title-Only Test Accuracy: {acc_test_title:.4f}")

In [ ]:
print("\nTraining Concatenated Model")
# Create the merged text column
train_data['merged_text'] = train_data['title'] + ' ' + train_data['content']
test_data['merged_text'] = test_data['title'] + ' ' + test_data['content']

# Preprocess the merged text column
train_data['merged_tokens'] = train_data['merged_text'].apply(preprocess_text).apply(add_bigrams_features)
test_data['merged_tokens'] = test_data['merged_text'].apply(preprocess_text).apply(add_bigrams_features)

# Train and evaluate
model_merged = NaiveBayes()
model_merged.fit(train_data, smoothening=1.0, class_col='label', text_col='merged_tokens')
model_merged.predict(train_data, text_col='merged_tokens', predicted_col='pred_train')
model_merged.predict(test_data, text_col='merged_tokens', predicted_col='pred_test')

acc_train_merged = model_merged.accuracy(train_data, class_col='label', predicted_col='pred_train')
acc_test_merged = model_merged.accuracy(test_data, class_col='label', predicted_col='pred_test')
print(f"Concatenated Model Test Accuracy: {acc_test_merged:.4f}")

In [ ]:
# Create a DataFrame for comparison
comparison_data = [
    {'Model': 'Content Only (Unigrams + Bigrams)', 
     'Training Accuracy': acc_train_content, 
     'Test Accuracy': acc_test_content},
    {'Model': 'Title Only (Unigrams + Bigrams)', 
     'Training Accuracy': acc_train_title, 
     'Test Accuracy': acc_test_title},
    {'Model': 'Title + Content (Concatenated)', 
     'Training Accuracy': acc_train_merged, 
     'Test Accuracy': acc_test_merged}
]

comparison_df = pd.DataFrame(comparison_data).set_index('Model')

print("\n\n--- Model Performance Comparison ---")
display(comparison_df.style.format('{:.4f}'))

In [ ]:
class DualParameterNB:
    def __init__(self):
        self.class_priors = None
        self.title_word_lhoods = None
        self.content_word_lhoods = None
        self.vocab, self.classes_, self.class_map_ = None, None, None
        
    def fit(self, df, smoothening, class_col='label', title_col='title_tokens', content_col='content_tokens'):
        self.smoothening = smoothening
        
        # Create class mapping
        self.classes_ = np.sort(df[class_col].unique())
        self.class_map_ = {label: i for i, label in enumerate(self.classes_)}
        num_classes = len(self.classes_)
        
        # Calculate class priors
        class_cnts = df[class_col].value_counts().reindex(self.classes_).fillna(0)
        self.class_priors = np.log(class_cnts / len(df))
        
        # Build vocabulary 
        print("Building shared vocabulary...")
        vocab_title = set(word for tokens_list in df[title_col] for word in tokens_list)
        vocab_content = set(word for tokens_list in df[content_col] for word in tokens_list)
        self.vocab = sorted(list(vocab_title | vocab_content), key=str)
        self.vocab_index = {word: i for i, word in enumerate(self.vocab)}
        vocab_size = len(self.vocab)
        
        # Tally word counts separately for title and content
        counts_title = np.zeros((num_classes, vocab_size))
        totals_title = np.zeros(num_classes)
        counts_content = np.zeros((num_classes, vocab_size))
        totals_content = np.zeros(num_classes)
        
        for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Fitting Model"):
            label = row[class_col]
            class_idx = self.class_map_[label]
            
            # Tally for title
            for token in row[title_col]:
                if token in self.vocab_index:
                    counts_title[class_idx, self.vocab_index[token]] += 1
            totals_title[class_idx] += len(row[title_col])

            # Tally for content
            for token in row[content_col]:
                if token in self.vocab_index:
                    counts_content[class_idx, self.vocab_index[token]] += 1
            totals_content[class_idx] += len(row[content_col])
            
        # Calculate smoothed log-likelihoods for each feature set
        num_title = counts_title + self.smoothening
        den_title = totals_title + self.smoothening * vocab_size
        self.title_word_lhoods = np.log(num_title / den_title[:, np.newaxis])
        
        num_content = counts_content + self.smoothening
        den_content = totals_content + self.smoothening * vocab_size
        self.content_word_lhoods = np.log(num_content / den_content[:, np.newaxis])

    def predict(self, df, title_col='title_tokens', content_col='content_tokens', predicted_col='Predicted'):
        predictions = []
        for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Predicting Classes"):
            log_posterior = self.class_priors.copy()
            
            # Add likelihoods from title tokens
            for token in row[title_col]:
                if token in self.vocab_index:
                    log_posterior += self.title_word_lhoods[:, self.vocab_index[token]]
            
            # Add likelihoods from content tokens
            for token in row[content_col]:
                if token in self.vocab_index:
                    log_posterior += self.content_word_lhoods[:, self.vocab_index[token]]
            
            predicted_index = np.argmax(log_posterior)
            predictions.append(self.classes_[predicted_index])
        
        df[predicted_col] = predictions

    def accuracy(self, df, class_col="label", predicted_col="Predicted"):
        return (df[class_col] == df[predicted_col]).sum() / len(df) if len(df) > 0 else 0

In [ ]:
# Preprocess Title and Content Separately
print("--- Preprocessing Title and Content Features ---")

train_data['title_tokens'] = train_data['title'].apply(preprocess_text).apply(add_bigrams_features)
test_data['title_tokens'] = test_data['title'].apply(preprocess_text).apply(add_bigrams_features)

train_data['content_tokens'] = train_data['content'].apply(preprocess_text).apply(add_bigrams_features)
test_data['content_tokens'] = test_data['content'].apply(preprocess_text).apply(add_bigrams_features)


# Train and Evaluate the Dual-Parameter Model
print("\n--- Training Dual-Parameter Model ---")
model_dual = DualParameterNB()
model_dual.fit(train_data, smoothening=1.0)
model_dual.predict(train_data, predicted_col='pred_train')
model_dual.predict(test_data, predicted_col='pred_test')

acc_train_dual = model_dual.accuracy(train_data, class_col='label', predicted_col='pred_train')
acc_test_dual = model_dual.accuracy(test_data, class_col='label', predicted_col='pred_test')

print(f"Dual-Parameter Model Test Accuracy: {acc_test_dual:.4f}")

In [ ]:
comparison_data = [
    {'Model': 'Content Only', 
     'Training Accuracy': acc_train_content, 
     'Test Accuracy': acc_test_content},
    {'Model': 'Title Only', 
     'Training Accuracy': acc_train_title, 
     'Test Accuracy': acc_test_title},
    {'Model': 'Title + Content (Concatenated)', 
     'Training Accuracy': acc_train_merged, 
     'Test Accuracy': acc_test_merged},
    {'Model': 'Title + Content (Dual Parameter)', 
     'Training Accuracy': acc_train_dual, 
     'Test Accuracy': acc_test_dual}
]

comparison_df = pd.DataFrame(comparison_data).set_index('Model')

print("\n\n--- Final Model Performance Comparison ---")
display(comparison_df.style.format('{:.4f}'))

In [ ]:
# Determine the number of unique classes
num_classes = train_data['label'].nunique()
print(f"Number of unique classes (K): {num_classes}")

# Calculate the random guessing accuracy
random_accuracy = 1 / num_classes
print(f"Random Guessing Accuracy: {random_accuracy:.4f} or {random_accuracy:.2%}")

In [ ]:
# Find the most frequent class in the training data
majority_class = train_data['label'].mode()[0]
print(f"Most frequent class in training data: {majority_class}")

# Calculate the accuracy by predicting this class for all test samples
majority_accuracy = (test_data['label'] == majority_class).sum() / len(test_data)
print(f"Majority Class Baseline Accuracy: {majority_accuracy:.4f} or {majority_accuracy:.2%}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = test_data['label']
y_pred = test_data['pred_test']

# Get the unique class labels from your trained model
class_labels = model_dual.classes_

# Calculate the confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_labels)

# Plot the confusion matrix as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, yticklabels=class_labels)

plt.title('Confusion Matrix for Dual-Parameter Model', fontsize=16)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

In [ ]:
# Find the index of the highest value on the diagonal
highest_diag_index = np.diag(cm).argmax()

# Get the corresponding class label
best_predicted_class = class_labels[highest_diag_index]
highest_value = np.diag(cm).max()

print(f"The category with the highest value on the diagonal is: Class '{best_predicted_class}'")
print(f"Number of correct predictions for this class: {highest_value}")